# Chicago, IL — Land Value Tax model

Models a revenue-neutral shift of the **City of Chicago's own property tax levy** from the
current value-based system toward land, under two reforms:

1. **4:1 split-rate** — land taxed at 4× the improvement rate (canonical cross-city export).
2. **Full building abatement** — improvements 100% exempt (pure land value tax), shown as a comparison.

## Policy decisions (confirmed before modeling)
| Decision | Choice |
|---|---|
| Government body | **City of Chicago levy only** (not the composite of all taxing agencies) |
| Reform | **Both** — 4:1 split-rate and 100% building abatement, compared |
| Exemptions | Preserve intent, but **modeled on gross EAV** — see limitations |
| Validation target | City of Chicago **Corporate** extension, TY2024 |

## Data sources (public, reproducible)
| Need | Cook County dataset | Fields |
|---|---|---|
| Land/building assessed values | **Assessor – Assessed Values** (`uzyt-m557`) | `board_land`, `board_bldg`, `board_tot` (+ certified/mailed fallback), `class`, `township_code` |
| City filter + census geometry | **Assessor – Parcel Universe** (`nj4t-kc8j`) | `lat`, `lon`, `tax_code`, `chicago_community_area_name` |

The eight Cook County assessment townships **70–77 are coterminous with the City of Chicago**;
the filter uses those township codes (≈ 883k parcels, matching the city's parcel count).

## Cook County assessment mechanics (TY2024)
- **Assessed value → market value**: Cook County assesses by class — residential/vacant/multi-family
  (major classes 1/2/3/9) at **10%** of market value, commercial/industrial (class 5) at **25%**,
  not-for-profit (class 4) at **20%**, incentive classes (6/7/8) at a reduced 10% in their active phase.
  The published values are *assessed* values; market value = assessed ÷ level-of-assessment.
- **Equalized Assessed Value (EAV)** = assessed × the IDOR Cook County state equalizer = **3.0355** (TY2024 final).
- The **City of Chicago Corporate** levy (TY2024) is **$1,642,587,611** at an agency rate of **1.495780%**
  (Cook County Clerk 2024 Tax Rate Report). Total Chicago EAV that year was $109.8B.

## Why the reform is computed on **market value**
The current system taxes EAV, which embeds Cook County's non-uniform class ratios (a $ of commercial
land is taxed 2.5× a $ of residential land). A land value tax taxes a dollar of land value uniformly,
so the split-rate / abatement are applied to **market-value** land and improvement (assessed ÷ class
level of assessment). The modeled change therefore reflects both the move to a uniform land base and
the land/building shift.

## Limitations
- **Exemptions not applied.** The public Assessed Values dataset carries no per-parcel homeowner/senior
  exemptions, so the model runs on **gross EAV**. The effective city rate derived below is therefore
  lower than the published 1.4958% (the same levy spread over a broader, pre-exemption base). Because
  the model is revenue-neutral and current tax is proportional to EAV, the *distribution* of winners and
  losers is robust to this; absolute rates are not exemption-exact.
- **TIF increment not removed** from the base (same reasoning).
- **Incentive classes (6/7/8)** are assumed to be at their 10% active-phase level; a minority in years
  11–12 (15%/20%) or expired would have market value slightly overstated. Small share of parcels.
- **TY2024 is the City of Chicago triennial reassessment year**, so Board of Review certification is
  still rolling out township-by-township. Each parcel uses the most final value available — Board where
  present, else certified, else mailed (`coalesce` below) — so a minority of parcels reflect a pre-Board
  stage. Values are final for the majority; this will settle as the reassessment completes.


## Section 1 — Imports and setup

In [1]:
import sys
import time
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')  # headless
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from shapely.geometry import Point
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    model_full_building_abatement,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

# ---- Constants -------------------------------------------------------------
CITY_NAME   = 'chicago'
STATE_FIPS  = '17'      # Illinois
COUNTY_FIPS = '031'     # Cook County
MODEL_TYPE  = 'split_rate:4.0'
LAND_IMPROVEMENT_RATIO = 4.0
TAX_YEAR    = 2024

# Cook County / City of Chicago parameters (TY2024)
EQUALIZER          = 3.0355          # IDOR Cook County final multiplier, TY2024
CITY_LEVY          = 1_642_587_611   # City of Chicago Corporate extension (Cook County Clerk 2024 Tax Rate Report)
PUBLISHED_CITY_RATE = 0.0149578      # City of Chicago Corporate agency rate (1.495780%)

# The 8 Cook County assessment townships coterminous with the City of Chicago
CHICAGO_TOWNSHIPS = ['70', '71', '72', '73', '74', '75', '76', '77']

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
print('setup OK')


setup OK


## Section 2 — Fetch / load parcel data

Downloads from the Cook County open-data Socrata API with pagination, filtered to the eight
Chicago townships and `TAX_YEAR`. Assessed values (`uzyt-m557`) are joined to the Parcel Universe
(`nj4t-kc8j`) for latitude/longitude (used to build the census geometry) and locational fields.
Cached to `data/parcels.gpq`.

In [2]:
SOCRATA = 'https://datacatalog.cookcountyil.gov/resource'

def fetch_socrata(resource, select, where, page=25000, max_pages=400, retries=5):
    # Page through a Socrata resource into a DataFrame, with retry/backoff
    # (the API can be slow/flaky for these large, wide tables).
    rows, offset = [], 0
    for _ in range(max_pages):
        params = {
            '$select': select,
            '$where': where,
            '$limit': page,
            '$offset': offset,
            '$order': ':id',
        }
        batch = None
        for attempt in range(retries):
            try:
                r = requests.get(f'{SOCRATA}/{resource}.json', params=params, timeout=120)
                r.raise_for_status()
                batch = r.json()
                break
            except Exception as e:
                if attempt == retries - 1:
                    raise
                time.sleep(2 ** attempt)   # exponential backoff
        if not batch:
            break
        rows.extend(batch)
        offset += page
        if len(batch) < page:
            break
        time.sleep(0.2)
    return pd.DataFrame(rows)

PARCEL_PATH = DATA_DIR / 'parcels.gpq'

if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(gdf):,} parcels from cache")
else:
    town_list = ','.join(f"'{t}'" for t in CHICAGO_TOWNSHIPS)

    # Assessed values (land / building / total), all three assessment stages
    av = fetch_socrata(
        'uzyt-m557',
        select=('pin,class,township_code,'
                'board_land,board_bldg,board_tot,'
                'certified_land,certified_bldg,certified_tot,'
                'mailed_land,mailed_bldg,mailed_tot'),
        where=f"year={TAX_YEAR} AND township_code in({town_list})",
    )
    print(f"Assessed values: {len(av):,} rows")

    # Parcel Universe: location + census geometry source
    pu = fetch_socrata(
        'nj4t-kc8j',
        select='pin,lat,lon,tax_code,chicago_community_area_name',
        where=f"year={TAX_YEAR} AND township_code in({town_list})",
    )
    print(f"Parcel universe: {len(pu):,} rows")

    df = av.merge(pu, on='pin', how='left')

    num_cols = ['board_land','board_bldg','board_tot','certified_land','certified_bldg',
                'certified_tot','mailed_land','mailed_bldg','mailed_tot','lat','lon']
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    # Point geometry from parcel centroid (used by the census block-group join)
    df = df[df['lat'].notna() & df['lon'].notna()].copy()
    gdf = gpd.GeoDataFrame(
        df, geometry=[Point(xy) for xy in zip(df['lon'], df['lat'])], crs='EPSG:4326'
    )
    gdf.to_parquet(PARCEL_PATH)
    print(f"Downloaded and cached {len(gdf):,} parcels")

print(gdf[['pin','class','board_land','board_bldg','board_tot']].head())


Loaded 882,904 parcels from cache
              pin class  board_land  board_bldg  board_tot
0  19151120251019   299       915.0     16834.0    17749.0
1  19151190381006   299       868.0     11132.0    12000.0
2  19151250350000   234      3661.0     24339.0    28000.0
3  19153260150000   203      3780.0     19220.0    23000.0
4  19154300260000   590     12263.0       896.0    13159.0


## Section 3 — Classify, normalize to market value, flag exempt

**Column mapping**

| Concept | Column | Notes |
|---|---|---|
| Land assessed value | `assessed_land` | coalesce board → certified → mailed |
| Improvement assessed value | `assessed_bldg` | coalesce board → certified → mailed |
| Total assessed value | `assessed_tot` | coalesce board → certified → mailed |
| Class / use code | `class` | Cook County 3-char class (e.g. `299`, `EX`) |
| Exempt flag | `full_exmp` | 1 for class `EX`/`RR` or zero/blank assessment |
| Level of assessment | `loa` | 0.10 / 0.20 / 0.25 by major class |
| Market land value | `market_land` | `assessed_land / loa` |
| Market improvement value | `market_bldg` | `assessed_bldg / loa` |

Levels of assessment and category labels follow the Cook County Assessor *Classifications of Real
Property* (revised 12/16/2024). A **Transportation - Parking** category captures automotive improvements
(5-22, 7-22, 8-22, 7-52), gasoline stations (classes ending in **-23**), and minor improvements (classes
ending in **-90**, including 1-90 and surface parking lots); these keep their major-class level of
assessment and are held out of the "$0 improvement → Vacant Land" override. Class **2-41** (vacant land
under common ownership with an adjacent residence) is categorized as Vacant Land but, as a Class 2
parcel, is taxed at the 10% residential level.

In [3]:
# --- Coalesce assessment stages: board (final) -> certified -> mailed ---
def coalesce(*cols):
    out = gdf[cols[0]].copy()
    for c in cols[1:]:
        out = out.where(out > 0, gdf[c])
    return out.fillna(0).clip(lower=0)

gdf['assessed_land'] = coalesce('board_land', 'certified_land', 'mailed_land')
gdf['assessed_bldg'] = coalesce('board_bldg', 'certified_bldg', 'mailed_bldg')
gdf['assessed_tot']  = coalesce('board_tot',  'certified_tot',  'mailed_tot')

# --- Level of assessment by major class (Cook County, TY2024) ---
def class_to_loa(c):
    c = str(c).strip().upper()
    if c in ('EX', 'RR') or c[:1] in ('0', ''):
        return np.nan            # exempt / railroad — excluded
    d = c[0]
    if d in ('1', '2', '3', '9'):
        return 0.10              # vacant, residential, multi-family, class-3 incentive
    if d == '4':
        return 0.20              # not-for-profit
    if d == '5':
        return 0.25              # commercial / industrial
    if d in ('6', '7', '8'):
        return 0.10              # incentive, active phase (documented approximation)
    return 0.10

gdf['loa'] = gdf['class'].map(class_to_loa)

# --- Property category (Cook County class -> standard LVTShift category) ---
_SFR_SUB = {'02','03','04','05','06','07','08','09','34','78'}

def classify(c):
    c = str(c).strip().upper()
    if c in ('EX', 'RR'):
        return 'Exempt'
    if len(c) < 3:
        return 'Other'
    mj, sub = c[0], c[1:]
    # Transportation - Parking: automotive improvements (5-22/7-22/8-22, 7-52), gasoline
    # stations (x-23), and minor improvements (x-90, incl. 1-90 and surface parking lots).
    # Checked first so it wins over the major-class buckets below.
    if sub in ('90', '23') or c in ('522', '722', '822', '752'):
        return 'Transportation - Parking'
    if sub == '00' or c == '241':           # 2-41 vacant land under common ownership w/ residence
        return 'Vacant Land'
    if mj == '1':
        return 'Vacant Land'
    if mj == '2':
        if sub in _SFR_SUB:                 return 'Single Family Residential'
        if c in ('210', '295'):             return 'Townhome / Rowhouse'
        if c in ('299', '297'):             return 'Condominium'
        if c == '211':                      return 'Small Multi-Family (2-4 units)'
        if c == '212':                      return 'Mixed Use'
        if c == '225':                      return 'Large Multi-Family (5+ units)'   # SRO
        if c in ('224', '239', '240'):      return 'Agricultural'
        return 'Other Residential'          # 201 garage, 213 co-op, 218/219 B&B, 236, 288
    if mj in ('3', '9'):                    # multi-family (and class-3 incentive)
        if c in ('318', '918'):             return 'Mixed Use'
        if c in ('399', '959'):             return 'Condominium'
        return 'Large Multi-Family (5+ units)'
    if mj == '4':
        return 'Other'                      # not-for-profit (taxable at 20%)
    if mj in ('5', '6', '7', '8'):          # commercial / industrial / incentive
        if mj == '6':                       return 'Industrial'                      # class 6 = industrial incentive
        if c in ('550','580','581','583','587','589','593'): return 'Industrial'     # 5B industrial
        if mj == '8' and c in ('880','881','893'):           return 'Industrial'
        if c in ('591','791','891','774'):                   return 'Office / Commercial Condo'
        if c in ('599','798','799','899','689'):             return 'Office / Commercial Condo'
        if sub in ('16','29','46','48'):    return 'Hotel'
        if sub == '92':                     return 'Mixed Use'
        if sub in ('17','26','27','28','30','31','32','35','01','53','60','61','65'):
            return 'Retail / General Commercial'
        return 'Other Commercial'
    return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf['class'].map(classify)

# --- Exempt flag: class EX/RR, or no taxable assessment ---
gdf['full_exmp'] = (
    gdf['class'].astype(str).str.upper().isin(['EX', 'RR'])
    | (gdf['assessed_tot'] <= 0)
    | gdf['loa'].isna()
).astype(int)

# --- Market values (assessed / level of assessment) ---
loa = gdf['loa'].fillna(0.10)
gdf['market_land'] = (gdf['assessed_land'] / loa).clip(lower=0)
gdf['market_bldg'] = (gdf['assessed_bldg'] / loa).clip(lower=0)

# Override: zero improvement -> Vacant Land (regardless of class), EXCEPT parking
# (surface lots often have $0 building value but should stay Transportation - Parking).
gdf.loc[(gdf['assessed_bldg'] <= 0) & (gdf['PROPERTY_CATEGORY'] != 'Transportation - Parking'),
        'PROPERTY_CATEGORY'] = 'Vacant Land'

print(f"Parcels: {len(gdf):,}   Fully-exempt (EX/RR/zero): {(gdf['full_exmp']==1).sum():,}")
print("\nCategory counts:")
print(gdf['PROPERTY_CATEGORY'].value_counts())
print("\nLevel-of-assessment distribution (taxable):")
print(gdf.loc[gdf['full_exmp']==0, 'loa'].value_counts(dropna=False))


Parcels: 882,904   Fully-exempt (EX/RR/zero): 52,735

Category counts:
Condominium                       287166
Single Family Residential         278520
Small Multi-Family (2-4 units)    120202
Vacant Land                        92560
Townhome / Rowhouse                23020
Mixed Use                          15962
Transportation - Parking           15957
Retail / General Commercial        13232
Industrial                         11404
Large Multi-Family (5+ units)      11258
Other Commercial                    4359
Other Residential                   4309
Office / Commercial Condo           4245
Hotel                                436
Other                                172
Exempt                               100
Agricultural                           2
Name: PROPERTY_CATEGORY, dtype: int64

Level-of-assessment distribution (taxable):
0.10    781295
0.25     48664
0.20       210
Name: loa, dtype: int64


## Section 4 — Current tax (City of Chicago levy only)

The City of Chicago Corporate levy is spread across all taxable equalized assessed value at a uniform
citywide agency rate. We compute gross EAV per parcel, then derive the effective city rate that
reproduces the official levy on this (pre-exemption) base. Revenue match to the official levy is exact
by construction; the meaningful check is that the **derived effective rate** is in the neighborhood of
the published 1.4958% — lower, because exemptions/TIF are not removed from the base.

In [4]:
# Gross EAV
gdf['eav'] = gdf['assessed_tot'] * EQUALIZER

taxable = gdf['full_exmp'] == 0
total_eav_taxable = float(gdf.loc[taxable, 'eav'].sum())

# Derive the effective city rate that reproduces the official City Corporate levy
city_rate = CITY_LEVY / total_eav_taxable
gdf['millage_rate'] = city_rate * 1000.0   # calculate_current_tax divides by 1000

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='eav',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

print(f"Taxable gross EAV:        ${total_eav_taxable:,.0f}")
print(f"Derived effective rate:    {city_rate*100:.4f}%   (published City Corporate: {PUBLISHED_CITY_RATE*100:.4f}%)")
print(f"Modeled current revenue:  ${current_revenue:,.0f}")
print(f"Official City levy:       ${CITY_LEVY:,.0f}")
print(f"Gap: {(current_revenue/CITY_LEVY - 1)*100:+.3f}%   (≈0 by construction)")

rate_ratio = city_rate / PUBLISHED_CITY_RATE
print(f"\nDerived/published rate ratio: {rate_ratio:.3f} "
      f"(<1 expected: gross EAV exceeds the net taxable base by exemptions + TIF)")


Taxable gross EAV:        $139,992,554,872
Derived effective rate:    1.1733%   (published City Corporate: 1.4958%)
Modeled current revenue:  $1,642,587,611
Official City levy:       $1,642,587,611
Gap: +0.000%   (≈0 by construction)

Derived/published rate ratio: 0.784 (<1 expected: gross EAV exceeds the net taxable base by exemptions + TIF)


## Section 5 — Split-rate model (4:1) on market value

Fully-exempt parcels are held out of the revenue-neutral solver and excluded from the export and report
charts (they carry no signal). The reform is solved on **market-value** land and improvement so a dollar
of land is taxed uniformly across classes.

In [5]:
# Hold out fully-exempt parcels (excluded from model, export, and charts)
n_exempt = int((gdf['full_exmp'] == 1).sum())
gdf = gdf[gdf['full_exmp'] == 0].copy()

# Taxable value columns expected by save_standard_export = the reform (market) base
gdf['taxable_land_value'] = gdf['market_land']
gdf['taxable_improvement_value'] = gdf['market_bldg']

land_millage, improvement_millage, new_revenue, gdf = model_split_rate_tax(
    df=gdf,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=gdf['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f"Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, and charts).")
print(f"Land millage: {land_millage:.5f}   Improvement millage: {improvement_millage:.5f}   "
      f"(ratio {land_millage/improvement_millage:.1f}:1)")
print(f"Revenue check: ${new_revenue:,.0f}  vs current ${gdf['current_tax'].sum():,.0f}")

category_summary = calculate_category_tax_summary(
    df=gdf, category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax', new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title=f"{CITY_NAME} — 4:1 split-rate tax impact")


Held out 52,735 fully-exempt parcels (excluded from model, export, and charts).
Land millage: 10.61294   Improvement millage: 2.65323   (ratio 4.0:1)
Revenue check: $1,642,587,611  vs current $1,642,587,611

chicago — 4:1 split-rate tax impact
                      Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
                   Condominium 287166     $16,942,325        5.7%        $59          $-4    7.7%      -1.2%            32.6%            29.7%
     Single Family Residential 278520    $121,708,296       37.5%       $437         $221   35.2%      30.8%            80.2%             0.6%
Small Multi-Family (2-4 units) 120202     $67,332,369       35.8%       $560         $262   28.3%      22.8%            71.4%             3.7%
                   Vacant Land  39925     $25,502,258      198.3%       $639         $198  372.6%     198.0%           100.0%             0.0%
           Townhome / Rowhouse  23020    

## Section 5b — Gate 5 artifact scan

Read the category table against economic priors before trusting it. Watch for: distinct categories
piled at one extreme value (ceiling clustering), implausible building shares, or large placeholder/zero
building counts. A condo unit with apportioned land is normal in Cook County; a built commercial
category showing ~0% building share is not.

In [6]:
d = gdf.copy()
d['bldg_share'] = d['market_bldg'] / (d['market_land'] + d['market_bldg']).replace(0, np.nan)
scan = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_zero_bldg=('market_bldg', lambda s: (s <= 1).mean()),
).sort_values('median_pct', ascending=False)
pd.set_option('display.width', 160)
print(scan.round(3))


                                     n  median_pct  median_bldg_share  pct_zero_bldg
PROPERTY_CATEGORY                                                                   
Vacant Land                      39925     197.976              0.000          0.998
Other Residential                 4309     179.901              0.081          0.000
Agricultural                         2      67.271              0.585          0.000
Single Family Residential       278520      30.817              0.748          0.000
Townhome / Rowhouse              23020      23.904              0.779          0.000
Small Multi-Family (2-4 units)  120202      22.825              0.784          0.000
Large Multi-Family (5+ units)    11258      16.930              0.810          0.000
Transportation - Parking         15957      15.911              0.055          0.004
Mixed Use                        15962      12.762              0.793          0.000
Industrial                       11404      -0.399              0

## Section 6 — Building abatement scenario (100% improvement exemption)

A full building abatement (pure land value tax), revenue-neutral to the same City levy, on the same
modeled (non-exempt) parcels. Shown for comparison; the canonical cross-city export remains the 4:1
split-rate.

In [7]:
gdf_abate = gdf.copy()
abate_millage, abate_revenue, gdf_abate = model_full_building_abatement(
    df=gdf_abate,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=gdf_abate['current_tax'].sum(),
    abatement_percentage=1.0,
)
print(f"Abatement land millage: {abate_millage:.5f}")
print(f"Revenue check: ${abate_revenue:,.0f}")

abate_summary = calculate_category_tax_summary(
    df=gdf_abate, category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax', new_tax_col='new_tax',
)
print_category_tax_summary(abate_summary, title=f"{CITY_NAME} — 100% building abatement (pure LVT)")

# Side-by-side median % change by category
cmp = pd.DataFrame({
    'split_4to1_median_pct': gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median(),
    'abatement_median_pct':  gdf_abate.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median(),
}).round(1)
print("\nMedian % change by category — split-rate vs abatement:")
print(cmp.sort_values('split_4to1_median_pct'))


Building abatement model (100.0% abatement)
Millage rate: 19.2833
Total tax revenue: $1,642,587,611.00
Target revenue: $1,642,587,611.00
Revenue difference: $0.00 (0.0000%)
Abatement land millage: 19.28330
Revenue check: $1,642,587,611

chicago — 100% building abatement (pure LVT)
                      Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
                   Condominium 287166    $-72,635,066      -24.4%      $-253        $-150  -19.7%     -41.2%            21.8%            70.0%
     Single Family Residential 278520    $170,789,793       52.6%       $613         $240   47.1%      36.4%            65.8%            21.7%
Small Multi-Family (2-4 units) 120202     $91,204,603       48.5%       $759         $181   30.4%      17.1%            55.0%            31.5%
                   Vacant Land  39925     $56,737,183      441.2%     $1,421         $442  441.3%     441.4%           100.0%             0.0%
   

## Section 7 — Census join + export

Census join must run before export. The canonical join uses parcel-centroid geometry to attach
block-group demographics. The split-rate result is the canonical cross-city export
(`analysis/data/chicago.csv` + `analysis/reports/chicago/`); the abatement scenario gets its own report
folder (`analysis/reports/chicago_abatement/`) but is **not** added to the cross-city dataset.

In [8]:
# Census join — must happen before export.
# Cook County has 4,002 census tracts; the default TIGERweb block-group fetch
# (get_census_data_with_boundaries) queries tract-by-tract and takes ~40 min here.
# Instead pull the whole Illinois block-group file once from the Census FTP (seconds)
# and filter to the county — same std_geoid + boundaries the canonical path produces.
from lvt.census_utils import (
    get_census_blockgroups_from_ftp, get_census_data, match_to_census_blockgroups,
)

_fips = STATE_FIPS + COUNTY_FIPS
try:
    census_gdf = get_census_blockgroups_from_ftp(_fips, 2022)   # one state zip, fast
    census_data = get_census_data(_fips, 2022)                  # ACS 5-year demographics
    census_gdf = census_gdf.merge(
        census_data, on='std_geoid', how='left', suffixes=('', '_census'))
    census_gdf = census_gdf.loc[:, ~census_gdf.columns.str.endswith('_census')]
    gdf = match_to_census_blockgroups(gdf, census_gdf)
    if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
        gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
    if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
        gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
    print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')


📥 Downloading block groups for state 17 from Census FTP...
✅ Downloaded 18,258,643 bytes
📊 Reading shapefile: tl_2022_17_bg.shp
🎯 Filtered to 4002 block groups in county 17031
Census join: 100.0% matched


In [9]:
# Propagate census columns onto the abatement frame (same index)
for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
    if _col in gdf.columns:
        gdf_abate[_col] = gdf[_col]

# Export (canonical) — split-rate 4:1
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/chicago/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)

# Building-abatement scenario — separate export (not in cross-city) + own report folder
abate_out = save_standard_export(
    df=gdf_abate,
    city=f'{CITY_NAME}_abatement',
    output_path=f'data/{CITY_NAME}_abatement.csv',
    model_type='abatement:100pct',
    land_millage=abate_millage,
    improvement_millage=0.0,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)
create_city_report(abate_out, f'{CITY_NAME}_abatement', show=False)
print("Done.")


  ✓ chicago: 830,169 rows → ../../analysis/data/chicago.csv  [model: split_rate:4.0]


/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


  ✓ chicago_abatement: 830,169 rows → data/chicago_abatement.csv  [model: abatement:100pct]
Done.


## Validation summary

| Check | Result |
|---|---|
| Revenue match | Current revenue set to the official City of Chicago Corporate levy $1,642,587,611 (TY2024, Cook County Clerk 2024 Tax Rate Report) — exact by construction |
| Effective rate | Derived effective city rate (see Section 4) vs published 1.4958%; difference reflects gross-EAV modeling (exemptions/TIF not removed) |
| Parcel count | ≈883k Chicago parcels; fully-exempt held out & excluded from charts |
| Census coverage | See Section 7 output |
| PNGs generated | `analysis/reports/chicago/` (split-rate) and `analysis/reports/chicago_abatement/` |
| Artifact scan (Gate 5) | See Section 5b |

*Figures populated on execution.*